# Week 1, Your First Investigation  (fully worked)

**What you'll do today.** Load a real dataset, ask it a question, and make your first chart,
a complete investigation end to end, before any theory. Then count culture in its other two
shapes: the words of a text and the pixels of an image corpus, the same move three times. Along the way you'll set up the one
ritual that carries the whole course: your Drive project folder.

The notebook has two halves. Everything through the brightness ranking runs in the
session; the sections from "Define a color" onward are the **after-class half**, about
twenty minutes, due before Week 2.

In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

## No installs today

Everything Week 1 needs ships with Colab already. (Later weeks have an install cell; today
the only setup is the Drive mount above, and, in class, putting your free Gemini key into
Colab Secrets for Week 7.)

In [ ]:
import io, os, re
import pandas as pd
import matplotlib.pyplot as plt
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"
print("SMOKE mode (tiny, offline):", SMOKE)

## Load real culture: 30 years of NYT wedding announcements

A CSV assembled by The Upshot: every NYT wedding announcement 1985–2014, whether the bride
kept or took a name, and the couple's ages. One `pd.read_csv(url)` and it's a table.

In [ ]:
SAMPLE = """url,name_status,bride_age,groom_age
http://www.nytimes.com/1985/12/01/style/myra-cohen-becomes-bride-of-a-rabbi.html,keeping,,
http://www.nytimes.com/1985/12/22/style/susan-stoddard-jacobsen-is-the-bride-of-john-colby-kean-3d-on-long-island.html,taking,,
http://www.nytimes.com/1990/04/15/style/louise-gilliam-weds-arthur-lee-mcgrady.html,taking,24,26
http://www.nytimes.com/1990/04/30/style/douglas-bernstein-weds-ellen-foley-fellow-actor.html,keeping,,
http://www.nytimes.com/1995/11/12/style/weddings-cameron-a-bruce-robert-masson-2d.html,taking,26,26
http://www.nytimes.com/1995/10/15/style/weddings-margaret-hart-alan-h-oshiki.html,keeping,34,38
http://www.nytimes.com/2000/02/06/style/weddings-amy-swartz-michael-rosen.html,taking,29,27
http://www.nytimes.com/2000/11/19/style/weddings-sherry-mcclaskey-brewster-thackeray.html,keeping,26,31
http://www.nytimes.com/2014/08/17/fashion/weddings/elizabeth-tepe-john-kneeland.html,taking,26,28
http://www.nytimes.com/2014/08/31/fashion/weddings/nancy-northup-james-johnson.html,keeping,55,53
"""
URL = "https://raw.githubusercontent.com/TheUpshot/nyt_weddings/master/nyt_wedding_announcements.csv"
LOCAL = "data/week01/nyt_weddings.csv"   # snapshot in the course repo; works offline

if os.path.exists(LOCAL):
    weddings = pd.read_csv(LOCAL)
    print("loaded the repo snapshot:", len(weddings), "announcements")
elif not SMOKE:
    try:
        weddings = pd.read_csv(URL)
        print("loaded from the web:", len(weddings), "announcements")
    except Exception as e:
        weddings = pd.read_csv(io.StringIO(SAMPLE))
        print("no network, using the small built-in sample:", e)
else:
    weddings = pd.read_csv(io.StringIO(SAMPLE))
    print("offline sample:", len(weddings), "announcements")
weddings.head()

## Ask it a question, and answer with a chart

Each announcement records whether the bride is *keeping* or *taking* a name, and the article
URL contains the date. Question: **did name-keeping become more common over these 30 years?**
The answer is three lines of code and one picture.

In [ ]:
weddings["year"] = weddings["url"].str.extract(r"nytimes\.com/(\d{4})/").astype(float)
by_year = (weddings[weddings["name_status"].isin(["keeping", "taking"])]
           .groupby("year")["name_status"]
           .apply(lambda s: (s == "keeping").mean()))

plt.figure(figsize=(7, 4))
plt.plot(by_year.index, by_year.values * 100, marker="o")
plt.ylabel("% of brides keeping their name")
plt.xlabel("year of announcement")
plt.title("Name-keeping in NYT wedding announcements")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "week01_first_chart.png"), dpi=150)
plt.show()
print("chart saved to your project folder")

**Before you trust it, one question the whole course turns on:** *whose* weddings are these?
NYT announcements skew wealthy, East-Coast, and socially prominent. The chart is real; the
"America" it describes is a slice. Naming that isn't a gotcha, it's the skill.

## When code errors (you will need this)

The AI will sometimes hand you code that errors. The move, printed on the cheat sheet
(`kits/common-errors-cheatsheet.md`), is: **read the last line of the traceback, paste it back
to the AI with "this errored, fix it and tell me what went wrong," try twice, then ask a
human.** Here's the drill on a bug you'll actually meet. This line looks right and isn't:

```python
by_year = weddings.groupby("Year")["name_status"].count()   # KeyError: 'Year'
```

The last line of that traceback says `KeyError: 'Year'`, the column is called `year`
(lowercase), and `weddings.columns` would have told you. Run the fixed version:

In [ ]:
# The fix: column names are exact. When a KeyError names a column, print the real ones.
print("the actual columns:", list(weddings.columns))
by_year_count = weddings.groupby("year")["name_status"].count()
by_year_count.head()

## Working with the AI assistant (guided)

In this course the AI writes most of the code; your job is to direct it and judge what
comes back. The loop, which you will run hundreds of times, has four steps:

1. **Ask precisely.** "Write a cell that counts announcements per year in `weddings` and
   draws a bar chart" — name the variable, name the output.
2. **Predict before you run.** One written sentence: what will this cell do? (This is the
   Week 1 check.)
3. **Run, then compare.** Did it do what you predicted? Where they differ, one of you is
   wrong, and finding out which is the learning.
4. **Interrogate.** Ask: *"explain this line by line"*, *"where would this break?"*,
   *"what is the smallest test that this did what I wanted?"* Then run that test.

Try the loop now. In Colab, open the assistant (the spark icon), give it the prompt in
step 1, and paste what it writes into the empty cell below. Write your prediction as a
comment first.

In [ ]:
# My prediction: this cell will ...
# (paste the assistant's code below, then run and compare)

# A reference version, if you want to check your result against one:
counts = weddings.groupby("year").size()
counts.plot(kind="bar", figsize=(7, 3), title="Announcements per year")
plt.tight_layout(); plt.show()

**Two rules that keep the loop honest.** Never run a cell you cannot say one sentence
about, and when a cell errors, read the last line of the traceback before pasting it back
to the assistant (the section above shows the recovery routine).

## Count a real corpus of words: Shakespeare's sonnets

For words, a corpus where every word was placed on purpose: the 154 sonnets of 1609,
public domain, snapshotted in `data/week01/`. Two lessons in one count: the stop-word
list is a choice, and it has to fit the corpus — a modern list leaves *thy* and *thou*
on top.

In [ ]:
import re
from collections import Counter

SONNET_18 = """Shall I compare thee to a summer's day?
Thou art more lovely and more temperate:
Rough winds do shake the darling buds of May,
And summer's lease hath all too short a date:
But thy eternal summer shall not fade,
Nor shall death brag thou wander'st in his shade,
So long as men can breathe, or eyes can see,
So long lives this, and this gives life to thee."""
SONNET_130 = """My mistress' eyes are nothing like the sun;
Coral is far more red, than her lips red:
If snow be white, why then her breasts are dun;
If hairs be wires, black wires grow on her head.
I have seen roses damask'd, red and white,
But no such roses see I in her cheeks;
And in some perfumes is there more delight
Than in the breath that from my mistress reeks.
I love to hear her speak, yet well I know
That music hath a far more pleasing sound:
I grant I never saw a goddess go;
My mistress, when she walks, treads on the ground:
    And yet by heaven, I think my love as rare,
    As any she belied with false compare."""

def load_sonnets():
    raw, src = None, "built-in pair (18 and 130)"
    if os.path.exists("data/week01/shakespeare_sonnets.txt"):
        raw, src = open("data/week01/shakespeare_sonnets.txt", encoding="utf-8-sig").read(), "repo snapshot"
    elif not SMOKE:
        try:
            import requests
            raw, src = requests.get("https://www.gutenberg.org/cache/epub/1041/pg1041.txt", timeout=30).text, "gutenberg.org"
        except Exception:
            pass
    if raw is None:
        return [SONNET_18, SONNET_130], src
    body = re.split(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", raw, flags=re.S)[-1]
    body = re.split(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", body)[0]
    parts = re.split(r"\n\s*([IVXLC]+)\s*\n", body)
    return [parts[i+1].strip() for i in range(1, len(parts) - 1, 2)
            if 200 < len(parts[i+1].strip()) < 900], src

sonnets, src = load_sonnets()
print(len(sonnets), "sonnets loaded from the", src)

sonnet_words = [w for s in sonnets for w in re.findall(r"[a-z']+", s.lower())]
print(len(sonnet_words), "words\n")
print("top words, raw:", Counter(sonnet_words).most_common(6))

STOP = set("the of a an to is in for and on with that not as it be by so but this all no nor if or when then what which".split())
kept = [w for w in sonnet_words if w not in STOP and len(w) > 2]
print("modern stop list: ", Counter(kept).most_common(6))

STOP |= set("thy thou thee thine doth dost art hath shall will i my me you your his her mine their more from are was were have has can may".split())
kept = [w for w in sonnet_words if w not in STOP and len(w) > 2]
print("stop list that fits the corpus:", Counter(kept).most_common(8))
# Three answers from one corpus. With the right stop list, the sonnets are about exactly
# what their reputation says: love (by a wide margin), then beauty, time, heart, sweet.
# The stop list is YOUR choice, and it has to be renegotiated for every corpus.

## Your turn: dials to turn

Every investigation below is one edited line plus a rerun. Change the value, predict the
chart, run, compare — the same loop as with the assistant, but the question is yours.

(The wedding articles' full text is paywalled and off-limits to scraping; the sanctioned
route, the official NYT API, is worked through in
`social-media-starters/nyt_announcement_text.ipynb`.)

In [ ]:
SONNET_WORD = "love"   # try: "time", "death", "black", "summer", "eyes"

import numpy as np
per_sonnet = [len(re.findall(rf"\b{SONNET_WORD}\b", s.lower())) for s in sonnets]
rolling = pd.Series(per_sonnet).rolling(15, min_periods=1).mean()
plt.figure(figsize=(7, 3))
plt.plot(range(1, len(sonnets) + 1), rolling, linewidth=2)
plt.xlabel("sonnet number"); plt.ylabel(f'"{SONNET_WORD}" per sonnet (rolling)')
plt.title(f'"{SONNET_WORD}" across the sequence of {len(sonnets)}')
plt.tight_layout(); plt.show()
# The sequence has real structure: try "black" and watch it spike after sonnet 127,
# where the dark lady sonnets begin. A word count finding the shape of a book.

In [ ]:
# A second dial, using columns we have not touched yet: the couple's ages.
ages = weddings.assign(
    bride=pd.to_numeric(weddings["bride_age"], errors="coerce"),
    groom=pd.to_numeric(weddings["groom_age"], errors="coerce"))
ages["gap"] = ages["groom"] - ages["bride"]
by_year = ages.groupby("year")[["bride", "groom", "gap"]].mean().dropna()
print(by_year.round(1).head(8))
by_year[["bride", "groom"]].plot(figsize=(7, 3), marker="o", markersize=3,
                                 title="Average ages at announcement")
plt.tight_layout(); plt.show()
# Your question here: does the age gap narrow? Do announced couples get older? Pick a
# claim, write it in one sentence, and check whether the chart supports it.

## Lab 2: count a real image corpus, pixel by pixel

Pictures are data the same way text is: a grid of red/green/blue numbers, so "how dark is
this painting?" is a counting question. The cell below loads real paintings from
the Met Museum's open-access collection (CC0): the course repo carries an 18-painting
snapshot in `data/week01/` (with a manifest crediting each), and the code falls back to
the live Met API, then to labeled panels, so it runs anywhere. In class the prepared
~200-thumbnail folder in your Drive plays this role; offline, colored panels stand in so the
counting code always runs.

In [ ]:
import io
import numpy as np
from PIL import Image

def swatch(rgb, size=96):
    arr = np.zeros((size, size, 3), dtype=np.uint8); arr[:, :] = rgb
    return Image.fromarray(arr)

def met_paintings(n=12, query="portrait", department=11):
    """Fetch n CC0 Met painting thumbnails; offline, colored panels stand in."""
    fallback = {"midnight": swatch((18, 18, 42)), "storm": swatch((70, 80, 95)),
                "forest": swatch((30, 90, 50)), "terracotta": swatch((150, 80, 60)),
                "gold": swatch((205, 160, 60)), "cream": swatch((235, 225, 205))}
    if os.path.exists("data/week01/met_manifest.csv"):
        man = pd.read_csv("data/week01/met_manifest.csv")
        return {r["title"][:26]: Image.open("data/week01/" + r["file"].split("data/week01/")[-1]
                if r["file"].startswith("data") else "data/week01/" + r["file"]).convert("RGB")
                for _, r in man.iterrows()}
    if SMOKE:
        return fallback
    try:
        import requests
        base = "https://collectionapi.metmuseum.org/public/collection/v1"
        ids = requests.get(f"{base}/search", timeout=20,
                           params={"hasImages": "true", "departmentId": department, "q": query}).json()["objectIDs"]
        images = {}
        for oid in ids:
            if len(images) >= n: break
            obj = requests.get(f"{base}/objects/{oid}", timeout=20).json()
            url = obj.get("primaryImageSmall")
            if not url: continue
            try:
                img = Image.open(io.BytesIO(requests.get(url, timeout=20).content)).convert("RGB")
                img.thumbnail((140, 140))
                images[obj.get("title", "untitled")[:26]] = img
            except Exception:
                continue
        if images: return images
    except Exception as e:
        print("no network, using the stand-in panels:", e)
    return fallback

corpus_images = met_paintings()
print(len(corpus_images), "images in the corpus")

In [ ]:
def avg_brightness(img):
    a = np.asarray(img).astype(float)
    return float((0.299*a[...,0] + 0.587*a[...,1] + 0.114*a[...,2]).mean())  # perceived luminance

def avg_color(img):
    return np.asarray(img).reshape(-1, 3).mean(axis=0) / 255

ranked = sorted(corpus_images, key=lambda k: avg_brightness(corpus_images[k]))
print("darkest -> brightest:")
for k in ranked:
    print(f"  {avg_brightness(corpus_images[k]):6.1f}  {k}")

fig, axes = plt.subplots(1, len(ranked), figsize=(1.3 * len(ranked), 2.6))
for ax, k in zip(np.atleast_1d(axes), ranked):
    ax.imshow(corpus_images[k]); ax.axis("off")
    ax.set_title(f"{avg_brightness(corpus_images[k]):.0f}", fontsize=8)
fig.suptitle("A real image corpus, ranked darkest to brightest by average luminance", fontsize=10)
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 3.2))
plt.bar(range(len(ranked)), [avg_brightness(corpus_images[k]) for k in ranked],
        color=[avg_color(corpus_images[k]) for k in ranked])
plt.xticks(range(len(ranked)), [k[:12] for k in ranked], rotation=45, ha="right", fontsize=7)
plt.ylabel("average brightness"); plt.tight_layout(); plt.show()

# The choice to notice: we counted BRIGHTNESS. Counting average hue, or saturation, or
# edge density would rank these paintings differently. What you choose to count of a
# picture is a decision, exactly like the stop-word list above.

In [ ]:
YOUR_QUERY = "landscape"   # try: "flowers", "armor", "dance", "night", "saint"

my_corpus = met_paintings(n=8, query=YOUR_QUERY)
if os.path.exists("data/week01/met_manifest.csv") and not SMOKE:
    print("note: the repo snapshot ignores the query; delete data/week01 or run in Colab for a live search")
my_ranked = sorted(my_corpus, key=lambda k: avg_brightness(my_corpus[k]))
fig, axes = plt.subplots(1, len(my_ranked), figsize=(1.3 * len(my_ranked), 2.4))
for ax, k in zip(np.atleast_1d(axes), my_ranked):
    ax.imshow(my_corpus[k]); ax.axis("off"); ax.set_title(f"{avg_brightness(my_corpus[k]):.0f}", fontsize=8)
fig.suptitle(f'"{YOUR_QUERY}", ranked by brightness', fontsize=10)
plt.tight_layout(); plt.show()

---

# After class: the second half (~20 minutes, before Week 2)

The session ends at the brightness ranking. These two sections are the take-home bridge
to Week 2: run them slowly, predict before running, and bring one observation to the next
session. They are the homework's guided half.

## Define a color, then count it

Brightness was one number per painting. A richer count needs a definition first: **what
counts as "red"?** A color range is a boundary you set on the hue wheel, and the boundary
is a choice — the pixel version of deciding what counts as a word. Define six named
ranges, count every pixel in every painting against them, and the corpus gets a palette.

In [ ]:
# The RANGES dict is yours to edit: rename a band, split the violets, add a golds band
# between yellows and oranges, and rerun everything below.
import matplotlib.colors as mcolors

RANGES = {"reds": (345, 15), "oranges/browns": (15, 45), "yellows": (45, 70),
          "greens": (70, 165), "blues": (165, 255), "violets": (255, 345)}
CHART_COLORS = {"reds": "#b03a2e", "oranges/browns": "#b9722f", "yellows": "#c9a227",
                "greens": "#3f6f5f", "blues": "#2e5f8a", "violets": "#6d4b7c",
                "neutrals": "#b4b2a9"}

def color_shares(img, sat_min=0.15, val_min=0.12, val_max=0.96):
    """Share of pixels in each named hue range; unsaturated or extreme pixels are neutrals."""
    hsv = mcolors.rgb_to_hsv(np.asarray(img).astype(float) / 255)
    hue = hsv[..., 0] * 360
    colored = (hsv[..., 1] >= sat_min) & (hsv[..., 2] >= val_min) & (hsv[..., 2] <= val_max)
    shares = {}
    for name, (lo, hi) in RANGES.items():
        in_range = ((hue >= lo) & (hue < hi)) if lo < hi else ((hue >= lo) | (hue < hi))
        shares[name] = float((in_range & colored).mean())
    shares["neutrals"] = float((~colored).mean())
    return shares

palette = pd.DataFrame({name: color_shares(im) for name, im in corpus_images.items()}).T
palette.plot(kind="bar", stacked=True, figsize=(8, 3.6),
             color=[CHART_COLORS[c] for c in palette.columns])
plt.ylabel("share of pixels"); plt.title("The corpus palette, counted against defined ranges")
plt.legend(fontsize=7, ncol=4); plt.xticks(rotation=45, ha="right", fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# The boundary is the analysis. Move the oranges/reds line by twenty degrees and
# recount: paintings change color category without changing a pixel.
wide_reds = dict(RANGES); wide_reds["reds"] = (325, 35); wide_reds["oranges/browns"] = (35, 45)

def reds_share(img, ranges):
    hsv = mcolors.rgb_to_hsv(np.asarray(img).astype(float) / 255)
    hue = hsv[..., 0] * 360
    colored = (hsv[..., 1] >= 0.15) & (hsv[..., 2] >= 0.12) & (hsv[..., 2] <= 0.96)
    lo, hi = ranges["reds"]
    in_range = ((hue >= lo) & (hue < hi)) if lo < hi else ((hue >= lo) | (hue < hi))
    return float((in_range & colored).mean())

for name, im in list(corpus_images.items())[:5]:
    a, b = reds_share(im, RANGES), reds_share(im, wide_reds)
    print(f"  {name:<26} reds: {a:5.1%} -> {b:5.1%}")
# "How red is this corpus?" has no answer until you say what red is. Same lesson as the
# stop-word list, in a different medium.

## What every bag throws away

Both counts today are "bags": the words with the order shaken out, the pixels with the
composition shaken out. That loss is the trade counting makes for scale, and it is worth
seeing on day one. Two demonstrations, ten seconds each.

In [ ]:
from collections import Counter

a = "the critics loved it and the public did not"
b = "the public loved it and the critics did not"
print("same bag?", Counter(a.split()) == Counter(b.split()))
print("same meaning? no - the bag cannot tell who loved and who did not\n")
c = "not good , actually bad"
d = "not bad , actually good"
print("same bag?", Counter(c.split()) == Counter(d.split()))
# Order is the first casualty, negation the second, who-did-what the third.

In [ ]:
# The centerpiece: Sonnet 130's bag versus Sonnet 130.
s130 = next((s for s in sonnets if "nothing like the sun" in s.lower()), SONNET_130)
bag = Counter(w for w in re.findall(r"[a-z']+", s130.lower()) if w not in STOP and len(w) > 2)
print("the bag of Sonnet 130:", [w for w, _ in bag.most_common(12)])
print("\nread as a bag: a catalogue of praise (sun, coral, roses, snow, music, goddess).")
print("now read the poem:\n")
print(s130)
# Every item in that catalogue is NEGATED: nothing like the sun, no such roses, music far
# more pleasing than her voice. The bag reads a conventional love poem; Shakespeare wrote
# a satire of conventional love poems. Negation, order, and argument are exactly what the
# bag cannot hold - and this is the largest possible version of the loss.

In [ ]:
# The same loss for pictures: a painting and its shuffled pixels are identical to
# every count we ran today - brightness, average color, the palette chart above.
if SMOKE:
    from PIL import ImageDraw
    pic = Image.new("RGB", (96, 96), (235, 225, 205))
    d = ImageDraw.Draw(pic)
    d.ellipse([18, 14, 78, 74], fill=(122, 59, 46)); d.rectangle([0, 78, 96, 96], fill=(63, 111, 95))
else:
    pic = corpus_images[ranked[len(ranked) // 2]].copy()

arr = np.asarray(pic).reshape(-1, 3)
rng = np.random.default_rng(0)
shuffled = Image.fromarray(arr[rng.permutation(len(arr))].reshape(np.asarray(pic).shape))

fig, axes = plt.subplots(1, 2, figsize=(5.6, 3))
for ax, im, title in zip(axes, [pic, shuffled], ["the painting", "its pixels, shuffled"]):
    ax.imshow(im); ax.axis("off")
    ax.set_title(f"{title}\nbrightness {avg_brightness(im):.1f}", fontsize=9)
plt.tight_layout(); plt.show()

# Identical bags, and only one is a picture. Composition is to images what word order is
# to sentences: the thing a bag cannot hold. Week 5's embeddings exist to recover some of
# what is lost here.

## You did a complete investigation

Loaded real culture, asked a question, answered it with a chart, questioned the chart's
scope, recovered from an error methodically, and counted culture in all three of its shapes: rows,
words, and pixels, the same move each time, with a choice hiding in each. Everything
else in this course is this loop with sharper tools, and it's all saved in your Drive
project folder, which will still be there next week.

**Sketch (homework):** one question from your own life you could answer with text or image
data; three sentences. No code.